In [0]:
# =====================================================
# PARAMÉTRAGE - Widgets pour exécution via Databricks Job
# =====================================================

dbutils.widgets.text("catalog_name", "banking_lakehouse", "Catalog Unity Catalog")
dbutils.widgets.text("environment", "dev", "Environnement (dev/staging/prod)")

CATALOG = dbutils.widgets.get("catalog_name")
ENVIRONMENT = dbutils.widgets.get("environment")

print(f"✅ Paramètres reçus : catalog={CATALOG}, environment={ENVIRONMENT}")

In [0]:
# =====================================================
# Notebook : 04_transform_silver_transactions
# Objectif : Nettoyer et fiabiliser les transactions
#            (Bronze -> Silver) en mode APPEND-ONLY
#            (pattern réaliste : une transaction ne se
#            met jamais à jour, elle est historique)
# Domaine  : Banking Lakehouse
# =====================================================

from pyspark.sql.functions import (
    col, current_timestamp, sha2, concat_ws, lit, when
)

# --- Configuration ---
TABLE_TRANSACTIONS_BRONZE = f"{CATALOG}.bronze.transactions_raw"
TABLE_TRANSACTIONS_SILVER = f"{CATALOG}.silver.transactions"

print("✅ Configuration Silver Transactions chargée")
print(f"Source Bronze : {TABLE_TRANSACTIONS_BRONZE}")
print(f"Cible Silver  : {TABLE_TRANSACTIONS_SILVER}")

In [0]:
# =====================================================
# Génération d'une clé technique (transaction_id)
# Version corrigée : hash sur TOUTES les colonnes numériques
# pour garantir l'unicité (déterministe et idempotent)
# + Application des règles de Data Quality
# =====================================================

df_bronze_transactions = spark.table(TABLE_TRANSACTIONS_BRONZE)

print(f"📊 Nombre de lignes en Bronze : {df_bronze_transactions.count()}")

# Liste de toutes les colonnes V1 à V28 + Time + Amount pour un hash robuste
feature_cols = [f"V{i}" for i in range(1, 29)] + ["Time", "Amount"]

df_transactions_quality = (
    df_bronze_transactions
    # --- Génération d'une clé technique via hash de TOUTES les features ---
    .withColumn(
        "transaction_id",
        sha2(concat_ws("||", *[col(c) for c in feature_cols]), 256)
    )
    
    # --- Flags qualité ---
    .withColumn(
        "dq_valid_amount",
        when(col("Amount") >= 0, lit(True)).otherwise(lit(False))
    )
    .withColumn(
        "dq_valid_time",
        when(col("Time") >= 0, lit(True)).otherwise(lit(False))
    )
    .withColumn(
        "dq_valid_class",
        when(col("Class").isin([0, 1]), lit(True)).otherwise(lit(False))
    )
    .withColumn(
        "dq_is_valid",
        col("dq_valid_amount") & col("dq_valid_time") & col("dq_valid_class")
    )
    
    .withColumn("_silver_processed_at", current_timestamp())
)

# --- Rapport de qualité ---
nb_total = df_transactions_quality.count()
nb_valid = df_transactions_quality.filter(col("dq_is_valid") == True).count()
nb_invalid = nb_total - nb_valid

print(f"\n📋 Rapport de Data Quality :")
print(f"📊 Total lignes         : {nb_total}")
print(f"✅ Lignes valides        : {nb_valid} ({round(100*nb_valid/nb_total, 4)}%)")
print(f"⚠️  Lignes avec anomalie : {nb_invalid} ({round(100*nb_invalid/nb_total, 4)}%)")

# --- Vérification unicité de la clé technique générée ---
nb_ids_distincts = df_transactions_quality.select("transaction_id").distinct().count()
print(f"\n🔎 Nombre de transaction_id distincts : {nb_ids_distincts} / {nb_total}")

if nb_ids_distincts == nb_total:
    print("✅ Unicité parfaite garantie !")
else:
    print(f"⚠️ Toujours {nb_total - nb_ids_distincts} collisions -> lignes réellement identiques en tout point")

display(df_transactions_quality.select(
    "transaction_id", "Time", "Amount", "Class", "dq_is_valid"
).limit(5))

In [0]:
# =====================================================
# Déduplication des doublons exacts détectés
# (1081 lignes identiques sur toutes les colonnes)
# Règle : on garde 1 seule occurrence par transaction_id
# =====================================================

from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

# On analyse d'abord la nature de ces doublons
print("🔎 Exemple de transaction_id dupliqué :")
duplicated_ids = (
    df_transactions_quality.groupBy("transaction_id")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
)
duplicated_ids.show(5)

# --- Déduplication : on garde une seule ligne par transaction_id ---
window_dedup_tx = Window.partitionBy("transaction_id").orderBy(col("_ingestion_timestamp").asc())

df_transactions_deduped = (
    df_transactions_quality
    .withColumn("_row_num", row_number().over(window_dedup_tx))
    .filter(col("_row_num") == 1)
    .drop("_row_num")
)

nb_avant = df_transactions_quality.count()
nb_apres = df_transactions_deduped.count()

print(f"\n📊 Lignes avant déduplication : {nb_avant}")
print(f"📊 Lignes après déduplication  : {nb_apres}")
print(f"🗑️  Doublons exacts supprimés   : {nb_avant - nb_apres}")

# Vérification finale d'unicité
nb_ids_finaux = df_transactions_deduped.select("transaction_id").distinct().count()
print(f"\n✅ Unicité finale : {nb_ids_finaux} IDs distincts / {df_transactions_deduped.count()} lignes")

In [0]:
# =====================================================
# Écriture en Silver - Mode APPEND-ONLY
# (les transactions sont historiques et immuables,
#  pas de MERGE nécessaire ici, contrairement aux clients)
# =====================================================

# On ne garde que les lignes valides (100% ici, mais on garde le filtre par principe)
df_final_transactions = df_transactions_deduped.filter(col("dq_is_valid") == True)

table_exists = spark.catalog.tableExists(TABLE_TRANSACTIONS_SILVER)

if not table_exists:
    print(f"🆕 Table {TABLE_TRANSACTIONS_SILVER} n'existe pas -> création initiale")
    df_final_transactions.write.format("delta").saveAsTable(TABLE_TRANSACTIONS_SILVER)
else:
    print(f"➕ Table {TABLE_TRANSACTIONS_SILVER} existe -> append incrémental")
    df_final_transactions.write.format("delta").mode("append").saveAsTable(TABLE_TRANSACTIONS_SILVER)

nb_final = spark.table(TABLE_TRANSACTIONS_SILVER).count()
print(f"\n✅ Table Silver Transactions prête : {nb_final} lignes")

# --- Vérification répartition fraude en Silver ---
print("\n📈 Répartition Class en Silver (post-nettoyage) :")
spark.table(TABLE_TRANSACTIONS_SILVER).groupBy("Class").count().orderBy("Class").show()